# 🧠 Reasoning & Thinking — pi.dev + Azure OpenAI

Notebook **07** of the pi.dev series. Some models can *think* before they answer,
producing a separate stream of reasoning tokens. `pi-ai` surfaces this as:

- a **`reasoning`** level you request per call (`"minimal" | "low" | "medium" | "high" | "xhigh"`),
- optional **`thinkingBudgets`** (per-level token caps, for token-based providers),
- **`thinking` content blocks** in the reply (distinct from `text` blocks), and
- streaming **`thinking_start` / `thinking_delta` / `thinking_end`** events, plus a
  `usage.reasoning` token count.

> ⚠️ **Reasoning is model-dependent.** Thinking output only appears if the underlying
> Azure deployment is a reasoning-capable model. On a plain chat deployment you'll get a
> normal answer with **no** `thinking` blocks — this notebook handles both gracefully and
> the token/cost accounting stays correct either way.

> **Kernel:** Deno.

## 1. Load env & register Azure OpenAI

We register the model with **`reasoning: true`** so `pi-ai` will send the reasoning request fields. If the deployment ignores them, you simply get a normal completion.

In [ ]:
import { loadEnvUp } from "playground/env";

const env = await loadEnvUp();
console.log(`\n${env.files.length} .env file(s) found; ${env.loaded.length} variable(s) loaded.`);

In [ ]:
import { registerAzure } from "playground/azure";

// `modelOverrides` is merged into every registered deployment — here we flip the
// model's `reasoning` capability flag on so pi-ai emits reasoning request fields.
const { models, model, modelId } = registerAzure({ modelOverrides: { reasoning: true } });
console.log(`Reasoning requested on "${modelId}" (effective only if the deployment supports it).`);

## 2. Stream a reasoning request

We ask a classic "trap" word problem — the kind where thinking step-by-step avoids the
intuitive-but-wrong answer. We pass **`{ reasoning: "medium" }`** as the stream option and
route the two channels separately:

- `thinking_*` events → a **🧠 thinking** channel
- `text_*` events → a **💬 answer** channel

You could also cap reasoning tokens per level with `thinkingBudgets`, e.g.
`{ reasoning: "medium", thinkingBudgets: { medium: 2048 } }` (token-based providers only).

In [ ]:
import type { Context } from "@earendil-works/pi-ai";

const reasoningContext: Context = {
  systemPrompt: "You are a careful reasoner. Think step by step, then give a concise final answer.",
  messages: [
    {
      role: "user",
      content:
        "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. " +
        "How much does the ball cost? Show your reasoning, then state the answer.",
      timestamp: Date.now(),
    },
  ],
};

const enc = new TextEncoder();
const write = (s: string) => { Deno.stdout.writeSync(enc.encode(s)); };

const reasoningStream = models.streamSimple(model, reasoningContext, { reasoning: "medium" });

let sawThinking = false;
for await (const ev of reasoningStream) {
  switch (ev.type) {
    case "thinking_start":
      sawThinking = true;
      write("\n🧠 thinking: ");
      break;
    case "thinking_delta":
      write(ev.delta);
      break;
    case "thinking_end":
      write("\n");
      break;
    case "text_start":
      write("\n💬 answer: ");
      break;
    case "text_delta":
      write(ev.delta);
      break;
    case "text_end":
      write("\n");
      break;
    case "done":
      write(`\n[done: ${ev.reason}]\n`);
      break;
    case "error":
      write(`\n[error: ${ev.error.errorMessage ?? "unknown"}]\n`);
      break;
  }
}

## 3. Inspect the finalized message

After the stream ends, `.result()` gives the assembled `AssistantMessage`. `thinking` and `text` are separate content blocks, and `usage.reasoning` reports how many of the output tokens were thinking tokens (when the provider breaks it down).

In [ ]:
const finalMsg = await reasoningStream.result();

const thinkingBlocks = finalMsg.content.filter((b) => b.type === "thinking");
const answerBlocks = finalMsg.content.filter((b) => b.type === "text");

console.log("=== 🧠 thinking blocks ===");
if (thinkingBlocks.length) {
  for (const b of thinkingBlocks) console.log((b as { thinking: string }).thinking);
} else {
  console.log(
    "(none — this deployment isn't reasoning-capable. The answer and usage below are still valid.)",
  );
}

console.log("\n=== 💬 answer blocks ===");
for (const b of answerBlocks) console.log((b as { text: string }).text);

console.log("\n=== usage ===");
console.log("stopReason:", finalMsg.stopReason);
console.log(
  `tokens: ${finalMsg.usage.input} in / ${finalMsg.usage.output} out` +
    (finalMsg.usage.reasoning !== undefined ? ` (of which ${finalMsg.usage.reasoning} reasoning)` : "") +
    `  |  cost: $${finalMsg.usage.cost.total.toFixed(6)}`,
);
console.log(`\nSaw streamed thinking events: ${sawThinking ? "yes ✅" : "no"}`);

## ✅ What just happened

1. We registered the model with **`reasoning: true`** so pi-ai sends reasoning request fields.
2. We streamed with **`{ reasoning: "medium" }`** and split the live output into a
   **🧠 thinking** channel (`thinking_*` events) and a **💬 answer** channel (`text_*` events).
3. From the finalized message we separated **`thinking`** vs **`text`** content blocks and read
   **`usage.reasoning`** — the thinking-token subset of `output`.
4. On a non-reasoning deployment there are simply no thinking blocks, and the notebook says so
   while still reporting valid usage/cost.

**Levels** trade latency/cost for depth: `minimal → low → medium → high → xhigh`. Cap tokens per
level with **`thinkingBudgets`** on token-based providers.

Next: **08 — Cost, caching & robustness** (usage/cost accounting, `cacheRetention`, `AbortSignal`, retries).